In [2]:
cd ..

/Users/hello/switchdrive/python/claude/diffraction


In [3]:
%matplotlib widget
import pylab as plt
import xrayutilities as xu
import numpy as np
from pathlib import Path
import copy
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure
import pickle
import swift
import roboticstoolbox as rtb
import spatialmath as sm
from diffraction import Crystal

In [3]:
%matplotlib widget
from diffraction import Crystal
import numpy as np
import swift
import roboticstoolbox as rtb

In [5]:
# Functions to be added to a python module

COLORS = {
    "O": [1,0,0,1],
    "Sr": [1,0.64,0.0,1],
    "Ti": [0.2,0.2,1,1],
}
class Crystal_Visualisation():
    def __init__(self, crystal):
        self.crystal = crystal
        self.multiplicity = []
        self.atoms = {}
        self.planes = {}
        self.scale=.1
        self.env=None

    def create_objects(self):
        multiplicity = []
        prevname = ""
        num=1
        for a, pos, occ, _ in self.crystal.xucalc.lattice.base():
            print(prevname, a.name)
            if a.name == prevname:
                num +=1
            else:
                prevname = a.name
                num=1
            mult = 0
            for i in range(-1, 2):
                for j in range(-1, 2):
                    for k in range(-1, 2):
                        atpos = (pos + [i, j, k])
                        if all(0 <= np.round(a,2) <= 1 for a in atpos):
                            vecpos = atpos[0]*self.crystal.xucalc.a1 + atpos[1]*self.crystal.xucalc.a2 + atpos[2]*self.crystal.xucalc.a3
                            print(f"{a.name}{num}_{mult}", np.round(np.array(vecpos),2))
                            mult+=1
                            if a.name in COLORS.keys():
                                atom_color = COLORS[a.name]
                            else:
                                atom_color = a.color
                            sphere = self.create_atom(
                                name=f"{a.name}{num}", 
                                position = (np.array(vecpos)+np.array([0,0,1]))*self.scale,
                                color = atom_color,
                                radius = a.radius/3,
                                scale = self.scale
                            )
                            self.atoms[f"{a.name}{num}_{mult}"] = sphere
                            
            multiplicity.append(mult)
        self.multiplicity = multiplicity
        
    def create_atom(self, name="atom", position=[0,0,0], color=(1,0,0,1), radius=None, scale=None):
        links, name, urdf_string, urdf_filepath = rtb.Robot.URDF_read(f"./atom.urdf", tld="/Users/hello/switchdrive/python/tools/atomic_motions")
        a = rtb.Robot(links, name=name, urdf_string=f'{urdf_string}', urdf_filepath=f'{urdf_filepath}',)
        a.links[-1].geometry[0].color = tuple(color)
        a.q = position
        a.q0 = position
        if radius is not None:
            a.links[-1].geometry[0].radius = radius
        if scale is not None:
            a.links[-1].geometry[0].radius = a.links[-1].geometry[0].radius*scale
        return a

    def show(self):
        if self.env is not None:
            self.env.reset()
        else:
            self.env = swift.Swift()
            self.env.launch(realtime=True)
        for a in self.atoms.values():
            self.env.add(a)

    def animate(self, vectors = [[]], amplitudes = [[]], dt=0.05, scale=1):
        for amplitudes_step in np.asarray(amplitudes).T:
            dq = np.sum([vec*a for vec, a in zip(vectors, amplitudes_step)], axis=0)
            for a, dqa in zip(self.atoms.values(), dq):
                a.q=a.q0+dqa*scale
            self.env.step(dt=dt)

    def add_white_planes(self, d=0):
        for n, plane in enumerate(["x", "y", "z"]):
            links, name, urdf_string, urdf_filepath = rtb.Robot.URDF_read(f"./plane_{plane}.urdf", tld="/Users/hello/switchdrive/python/tools/atomic_motions")
            a = rtb.Robot(links, name=name, urdf_string=f'{urdf_string}', urdf_filepath=f'{urdf_filepath}',)
            self.planes[n] = a
            self.env.add(a)
        self.set_plane_distance(d)

    def set_plane_distance(self, d):
        for n, a in self.planes.items():
            if n<2:
                a.q[n] = d
        self.env.step(0.05)

In [6]:
# Load a crystal from CIF file
crystal = Crystal.from_cif("SrTiO3", "diffraction/data/SrTiO3.cif")

# Set orientation relative to laboratory space and calc UB matrix
crystal.add_orientation((0,0,1), (0,0,1), "normal")
crystal.add_orientation((1,0,0), (0,1,0), "inplane along beam")
crystal.calc_ub()

# Set angular constraints
crystal.set_constraints(delta=0, phi=0, eta=0)

print(crystal)

### LATTICE ###
name  : SrTiO3
a     : 3.94513
b     : 3.94513
c     : 3.94513
alpha : 90.0
beta  : 90.0
gamma : 90.0

### UNIT CELL ###
0 (Sr (38), ('1a', (np.float64(0.0), np.float64(0.0), np.float64(0.0))), 1.0, 0.0)
1 (Ti (22), ('1a', (np.float64(0.5), np.float64(0.5), np.float64(0.5))), 1.0, 0.0)
2 (O ( 8), ('1a', (np.float64(0.5), np.float64(0.0), np.float64(0.5))), 1.0, 0.0)
3 (O ( 8), ('1a', (np.float64(0.5), np.float64(0.5), np.float64(0.0))), 1.0, 0.0)
4 (O ( 8), ('1a', (np.float64(0.0), np.float64(0.5), np.float64(0.5))), 1.0, 0.0)

### ORIENTATIONS ###
hkl            xyz            tag
(0, 0, 1)      (0, 0, 1)      normal
(1, 0, 0)      (0, 1, 0)      inplane along beam

### CONSTRAINTS ###
eta   : 0
phi   : 0
delta : 0



### Show the unit cell

In [7]:
# Create the atoms based on the unit cell of the crystal and add them to the visualization

vis = Crystal_Visualisation(crystal)
vis.create_objects()

 Sr
Sr1_0 [0. 0. 0.]
Sr1_1 [0.   0.   3.95]
Sr1_2 [0.   3.95 0.  ]
Sr1_3 [0.   3.95 3.95]
Sr1_4 [3.95 0.   0.  ]
Sr1_5 [3.95 0.   3.95]
Sr1_6 [3.95 3.95 0.  ]
Sr1_7 [3.95 3.95 3.95]
Sr Ti
Ti1_0 [1.97 1.97 1.97]
Ti O
O1_0 [1.97 0.   1.97]
O1_1 [1.97 3.95 1.97]
O O
O2_0 [1.97 1.97 0.  ]
O2_1 [1.97 1.97 3.95]
O O
O3_0 [0.   1.97 1.97]
O3_1 [3.95 1.97 1.97]


In [8]:
# Show the unit cell
vis.show()

In [7]:
# Add a white base plane
vis.add_white_planes()

## Example for an animation along two displacement vectors

### Create exemplary displacement vectors along x and y directions

In [8]:
displacement_vector_x = {
    "Sr1": [0.01, 0, 0],
    "Ti1": [0.03, 0, 0],
    "O1": [-0.1, 0, 0],
    "O2": [-0.1, 0, 0],
    "O3": [-0.1, 0, 0],
}

vector_x = []
# Take care of multiplicity
for (atom, displacement), mult in zip(displacement_vector_x.items(), vis.multiplicity):
    for n in range(mult):
        vector_x.append(displacement)
vector_x = np.array(vector_x)

displacement_vector_y = {
    "Sr1": [0, 0.01, 0],
    "Ti1": [0, 0.03, 0],
    "O1": [0, -0.1, 0],
    "O2": [0, -0.1, 0],
    "O3": [0, -0.1, 0],
}

vector_y = []
# Take care of multiplicity
for (atom, displacement), mult in zip(displacement_vector_y.items(), vis.multiplicity):
    for n in range(mult):
        vector_y.append(displacement)
vector_y = np.array(vector_y)

### Create exemplary displacement amplitudes

In [9]:
n = np.linspace(0,20*np.pi,500)
amplitudes = 0.5* np.array([np.sin(n), np.cos(n)])

### Start animation

In [10]:
vis.animate(vectors=[vector_x, vector_y], amplitudes=amplitudes, dt=0.01, scale=1)